# Student Score Prediction Case Study
## Multiple Linear Regression & Machine Learning Pipeline

### Problem Statement:
A university academic board wants to understand what factors drive student performance on final examinations. Specifically, they want to predict a student's continuous **final score (0 to 100)** before the final examination takes place, using indicators such as:
- **Attendance percentage**
- **Weekly self-study hours**
- **Continuous assignment average**
- **Midterm examination score**
- **Prior cumulative GPA**

We will build a Multiple Linear Regression model and follow the exact industry-standard workflow:
1. Reading and Understanding the Data
2. Data Inspection & Cleaning
3. Exploratory Data Analytics (EDA)
4. Data Preparation (Train/Test Split & Feature Rescaling)
5. Model Building & Multicollinearity (VIF)
6. Residual Analysis
7. Model Evaluation on Test Data & Fitted Equation

In [1]:
# Supress Warnings
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

## Step 1: Reading and Understanding the Data

In [2]:
# Load canonical dataset
students = pd.read_csv('../data/interim/students_clean.csv')
students.head()

## Step 2: Data Inspection & Cleaning

In [3]:
print("Shape of dataset:", students.shape)
print("\nDataset Information:")
print(students.info())
print("\nMissing values count:")
print(students.isnull().sum())
print("\nSummary Statistics:")
students.describe().round(2)

## Step 3: Exploratory Data Analytics (EDA)

### 3.1 Visualising Numeric Variables
Let us create pairplots to inspect pairwise relationships between all features and the target `final_score`.

In [4]:
numeric_features = ['attendance_pct', 'study_hours_week', 'assignment_avg', 'midterm_score', 'previous_gpa', 'final_score']
sns.pairplot(students[numeric_features])
plt.show()

### 3.2 Correlation Heatmap
Let us evaluate the Pearson correlation matrix to identify strong predictors and check for multicollinearity.

In [5]:
plt.figure(figsize=(8, 6))
corr = students[numeric_features].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", mask=mask, linewidths=0.5)
plt.title("Correlation Heatmap", fontsize=13, fontweight='bold')
plt.show()

### 3.3 Visualising Key Predictors vs. Target

In [6]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Midterm vs Final
sns.regplot(data=students, x='midterm_score', y='final_score', scatter_kws={'alpha': 0.6}, line_kws={'color': 'red'}, ax=ax1)
ax1.set_title("Midterm Score vs. Final Score (Strongest Predictor)")

# Study Hours vs Final
sns.regplot(data=students, x='study_hours_week', y='final_score', scatter_kws={'alpha': 0.6, 'color': 'purple'}, line_kws={'color': 'orange'}, ax=ax2)
ax2.set_title("Study Hours vs. Final Score")

plt.tight_layout()
plt.show()

## Step 4: Data Preparation

### 4.1 Splitting the Data into Training and Testing Sets
We split the data into **80% training** and **20% testing** using fixed `random_state=42` to guarantee reproducibility and eliminate test data leakage.

In [7]:
from sklearn.model_selection import train_test_split

# Feature matrix X and target y
X = students[['attendance_pct', 'study_hours_week', 'assignment_avg', 'midterm_score', 'previous_gpa']]
y = students['final_score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape:  {X_test.shape}")

### 4.2 Rescaling the Features
To put all features on a standardized scale (mean = 0, variance = 1), we fit `StandardScaler` **exclusively on the training data** and then transform both train and test.

In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
# Fit ONLY on train
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

X_train_scaled.describe().round(2)

## Step 5: Model Building

### 5.1 Multiple Linear Regression via SciKit-Learn

In [9]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

print(f"Intercept: {lr.intercept_:.4f}")
coef_table = pd.DataFrame({
    'Feature': X_train.columns,
    'Standardized Coefficient (Beta)': lr.coef_.round(4)
}).sort_values('Standardized Coefficient (Beta)', ascending=False)
coef_table

### 5.2 Checking Multicollinearity with Variance Inflation Factor (VIF)
VIF measures how much the variance of an estimated regression coefficient increases if predictors are correlated. Rule of thumb: VIF $< 5.0$ indicates safe, low collinearity.

In [10]:
from numpy.linalg import inv

corr_matrix = X_train_scaled.corr().values
inv_corr = inv(corr_matrix)
vif_data = []
for i, col in enumerate(X_train.columns):
    vif_data.append({'Feature': col, 'VIF': round(float(inv_corr[i, i]), 3)})
vif_df = pd.DataFrame(vif_data).sort_values('VIF', ascending=False)
vif_df

## Step 6: Residual Analysis of the Training Data
A major assumption of linear regression is that error terms (residuals) are normally distributed with zero mean and constant variance.

In [11]:
y_train_pred = lr.predict(X_train_scaled)
res_train = y_train - y_train_pred

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Residual Histogram
sns.histplot(res_train, kde=True, color="teal", bins=15, ax=ax1)
ax1.set_title("Distribution of Error Terms (Residuals)", fontsize=12, fontweight='bold')
ax1.set_xlabel("y_train - y_train_pred")

# Residual vs Fitted
sns.scatterplot(x=y_train_pred, y=res_train, color="blue", alpha=0.6, ax=ax2)
ax2.axhline(0, color='red', linestyle='--')
ax2.set_title("Residuals vs. Fitted Values (Homoscedasticity)", fontsize=12, fontweight='bold')
ax2.set_xlabel("Fitted Values")
ax2.set_ylabel("Residuals")

plt.tight_layout()
plt.show()

## Step 7: Model Evaluation on Test Data
Now we evaluate the model on the unseen test set.

In [12]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

y_test_pred = lr.predict(X_test_scaled)

r2_test = r2_score(y_test, y_test_pred)
rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
mae_test = mean_absolute_error(y_test, y_test_pred)

print(f"Holdout Test Set Evaluation:")
print(f"  R-squared (R2):                  {r2_test:.4f}")
print(f"  Root Mean Squared Error (RMSE):  {rmse_test:.3f}")
print(f"  Mean Absolute Error (MAE):       {mae_test:.3f}")

### 7.1 Plotting Actual vs. Predicted on Test Set

In [13]:
plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_test_pred, color='#0284c7', alpha=0.8)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='red', linestyle='--', linewidth=2, label='Perfect 45° Fit Line')
plt.title(f"Actual vs. Predicted Final Scores (R2 = {r2_test:.3f})", fontsize=13, fontweight='bold')
plt.xlabel("Actual Final Score", fontsize=11)
plt.ylabel("Predicted Final Score", fontsize=11)
plt.legend()
plt.show()

### 7.2 Equation of the Best Fitted Line
Converting standardized coefficients back to unscaled units so educators can interpret real point values:

In [14]:
unscaled_coefs = lr.coef_ / scaler.scale_
unscaled_intercept = lr.intercept_ - (unscaled_coefs * scaler.mean_).sum()

print("=" * 70)
print("FINAL FITTED REGRESSION EQUATION:")
print(f"final_score = {unscaled_intercept:.2f}")
for feat, coef in zip(X.columns, unscaled_coefs):
    print(f"            + ({coef:.4f} * {feat})")
print("=" * 70)